## 越南语图像内容搜索

模型: FacebookAI/xlm-roberta-base

框架: MindSpore + MindNLP

运行环境: 香橙派 AIpro（20T） + Ubuntu + MindSpore 2.5.0 + MindNLP 0.4.1

模型加载方式: HuggingFace 在线下载

任务目标: 输入越南语查询文本，检索语义最相关的图像及其越南语描述

In [1]:
!python -V
!pip show mindnlp

Python 3.9.2
Name: mindnlp
Version: 0.4.1
Summary: An open source natural language processing research tool box. Git version: [sha1]:22221f40, [branch]: (HEAD -> master, tag: v0.4.1, origin/master)
Home-page: https://github.com/mindlab-ai/mindnlp/tree/master/
Author: MindSpore Team
Author-email: 
License: Apache 2.0
Location: /home/HwHiAiUser/.local/lib/python3.9/site-packages
Requires: addict, datasets, evaluate, mindspore, ml-dtypes, pillow, pyctcdecode, pytest, regex, requests, safetensors, sentencepiece, tokenizers, tqdm
Required-by: 


In [2]:
import mindspore as ms

print(f"MindSpore 版本: {ms.__version__}")
print(f"当前运行设备: {ms.get_context('device_target')}")

[WARNING] ME(9563:255086326722592,MainProcess):2026-04-22-01:14:23.596.968 [mindspore/run_check/_check_version.py:324] MindSpore version 2.5.0 and Ascend AI software package (Ascend Data Center Solution)version 7.7 does not match, the version of software package expect one of ['7.5', '7.6']. Please refer to the match info on: https://www.mindspore.cn/install
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, ge

MindSpore 版本: 2.5.0
当前运行设备: Ascend


In [3]:
# 日志配置

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

### 下载万卷丝路图文数据集

安装必要的库（没有特定版本限制）

In [4]:
!pip install openxlab
!pip install -U openxlab 

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 kB 807.1 kB/s eta 0:00:001m784.7 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.1/953.1 kB 4.3 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.4/239.4 kB 4.3 MB/s eta 0:00:000:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 5.2 MB/s eta 0:00:000:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 k

### 配置与查看可用数据集

In [5]:
# 下载数据集
import openxlab
# 登录，进入网站 https://opendatalab.com/OpenDataLab/WanJuanSiLu2O
# 登录帐号后获取对应的个人 ak 和 sk，此演示中使用的 ak 和 sk 已销毁
openxlab.login(ak='74yxbpwowl4w4kdp5zjr', sk='2wgdvxmzjow7mxa9a0wmgwgkkb0pg6eozqakl1rk')

# 获取数据集信息及文件列表查看
from openxlab.dataset import info
info(dataset_repo='OpenDataLab/WanJuanSiLu2O')

2026-04-22 01:14:54,765 [WARNING] AK and SK have been configured. You can set relogin as true to force a relogin.


+-----------+-----------------------------------------------------------------------------------------------------+
| Field     | Content                                                                                             |
+-----------+-----------------------------------------------------------------------------------------------------+
| Name      | OpenDataLab/WanJuanSiLu2O                                                                           |
+-----------+-----------------------------------------------------------------------------------------------------+
| Introduct | 全新升级的“万卷·丝路2.0”，带来以下三大核心提升：                                                    |
| ion       | 语种数量显著扩充、数据模态全面升级，为                                                            8 |
|           | 个语种均提供了丰富的图片-文本、音频-文本、视频-文本、特色指令微调SFT四大模态数据，覆盖多...         |
+-----------+-----------------------------------------------------------------------------------------------------+
| Author    | Shanghai Artificial Intelligence Laboratory                                                         |
+-----------+-----------------------------------------------------------------------------------------------------+
| Data Type | Video, Image, Audio, Text                                                                           |
+-----------+-----------------------------------------------------------------------------------------------------+
| Label     | Text, Class labels                                                                                  |
| Type      |                                                                                                     |
+-----------+-----------------------------------------------------------------------------------------------------+
| Task Type | Natural Language Generation, Commonsense Reasoning, Code Generation                                 |
+-----------+-----------------------------------------------------------------------------------------------------+
| File List |  Total Size: 87.1G                                                                                  |
|           |  +----------------------------------+---------+                                                     |
|           |  | File                             |    Size |                                                     |
|           |  +----------------------------------+---------+                                                     |
|           |  | - /                              |         |                                                     |
|           |  +----------------------------------+---------+                                                     |
|           |  |    - README.md                   |   7.46K |                                                     |
|           |  +----------------------------------+---------+                                                     |
|           |  |    - metafile.yaml               | 445.00B |                                                     |
|           |  +----------------------------------+---------+                                                     |
|           |  | - /raw/                          |         |                                                     |
|           |  +----------------------------------+---------+                                                     |
|           |  |    - audio/ar/audio/ar_part1.zip |   2.00G |                                                     |
|           |  +----------------------------------+---------+                                                     |
|           |  |    - audio/ar/audio/ar_part2.zip |   1.99G |                                                     |
|           |  +----------------------------------+---------+                                                     |
|           |  |    - audio/ar/audio/ar_part3.zip |   1.99G |                                                     |
|           |  +-----

{'Name': 'OpenDataLab/WanJuanSiLu2O',
 'Introduction': '全新升级的“万卷·丝路2.0”，带来以下三大核心提升：\n语种数量显著扩充、数据模态全面升级，为 8 个语种均提供了丰富的图片-文本、音频-文本、视频-文本、特色指令微调SFT四大模态数据，覆盖多...',
 'Author': 'Shanghai Artificial Intelligence Laboratory',
 'Data Type': 'Video, Image, Audio, Text',
 'Label Type': 'Text, Class labels',
 'Task Type': 'Natural Language Generation, Commonsense Reasoning, Code Generation',
 'File List': {'Total size': '87.1G',
  '/': {'README.md': '7.46K', 'metafile.yaml': '445.00B'},
  '/raw/': {'audio/ar/audio/ar_part1.zip': '2.00G',
   'audio/ar/audio/ar_part2.zip': '1.99G',
   'audio/ar/audio/ar_part3.zip': '1.99G',
   'audio/ar/audio/ar_part4.zip': '1.99G',
   'audio/ar/audio/ar_part5.zip': '2.00G',
   'audio/ar/audio/ar_part6.zip': '1.95G',
   'audio/ar/audio/ar_part7.zip': '1.77G',
   'audio/ar/audio/ar_part8.zip': '1.67G',
   '...\n  (Showing 8 of 70 files)': ''}}}

In [6]:
# 下载路径配置
from openxlab.dataset import download

# 设置仓库名及文件相对路径
dataset_repo = 'OpenDataLab/WanJuanSiLu2O'
# 选择需要下载的数据集文件
source_file = '/raw/image/vi/vi_image_text_pair.jsonl'
save_path='./'

# 执行下载
download(dataset_repo=dataset_repo, source_path=source_file, target_path=save_path)

logging.info(f'下载完成')

2026-04-22 01:15:03,396 [INFO] 开始执行download命令，数据集仓库: OpenDataLab/WanJuanSiLu2O，源路径: /raw/image/vi/vi_image_text_pair.jsonl
2026-04-22 01:15:03,412 [INFO] 文件将保存到: /home/HwHiAiUser/OpenDataLab
2026-04-22 01:15:03,414 [INFO] 处理后的源路径: /raw/image/vi/vi_image_text_pair.jsonl


开始执行download命令，数据集仓库: OpenDataLab/WanJuanSiLu2O，源路径: /raw/image/vi/vi_image_text_pair.jsonl
文件将保存到: /home/HwHiAiUser/OpenDataLab


Fetching the list of files...

2026-04-22 01:15:03,459 [INFO] 开始获取文件列表
2026-04-22 01:15:03,464 [INFO] 正在获取文件列表，cursor: None
2026-04-22 01:15:03,807 [INFO] 文件列表获取完成，共 1 个文件
2026-04-22 01:15:03,813 [INFO] 执行下载前检查，文件路径: raw/image/vi/vi_image_text_pair.jsonl


文件列表获取完成，共 1 个文件


...

Total progress: 100%, total files:1/1, downloaded size: 35.0M, total size: 35.0M

2026-04-22 01:15:04,930 [INFO] 所有 1 个文件处理完成，总大小: 34.98M
2026-04-22 01:15:05,157 [INFO] 文件下载完成，保存路径: /home/HwHiAiUser/OpenDataLab/OpenDataLab___WanJuanSiLu2O/raw/image/vi/vi_image_text_pair.jsonl


Download Completed.

The file has been successfully downloaded to 
/home/HwHiAiUser/OpenDataLab/OpenDataLab___WanJuanSiLu2O/raw/image/vi/vi_image_text_pair.jsonl

2026-04-22 01:15:05,183 [INFO] 下载完成


In [7]:
# 读取数据集
import json
# 配置数据集路径，见上方日志倒数第二行
JSONL_PATH = '/home/HwHiAiUser/OpenDataLab/OpenDataLab___WanJuanSiLu2O/raw/image/vi/vi_image_text_pair.jsonl'

logging.info(f"读取数据集: {JSONL_PATH}")

all_data = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue

        img_id = record.get('img_id', '')
        image = record.get('image', {})
        caption = record.get('captions', {})
        labels = record.get('labels', {})

        url = image.get('path', '')   if isinstance(image, dict)   else ''
        content = caption.get('content', '') if isinstance(caption, dict) else ''
        level1 = labels.get('pjwk_cates', {}).get('level1', []) if isinstance(labels, dict) else []

        if not url or not content:
            continue

        all_data.append({
            'img_id':  img_id,
            'url':     url,
            'caption': content,
            'label':   level1[0] if level1 else 'unknown',
        })

logging.info(f"数据读取完成，共 {len(all_data)} 条")

# 样例展示
print(f"\n数据样例（前5条）:")
for item in all_data[:5]:
    print(f"\n [{item['label']}]")
    print(f"Caption: {item['caption'][:80]}...")
    print(f"URL: {item['url'][:60]}...")

2026-04-22 01:15:14,847 [INFO] 读取数据集: /home/HwHiAiUser/OpenDataLab/OpenDataLab___WanJuanSiLu2O/raw/image/vi/vi_image_text_pair.jsonl
2026-04-22 01:15:19,328 [INFO] 数据读取完成，共 77486 条



数据样例（前5条）:

 [场景类]
Caption: Đến năm 2025 cả nước sẽ có khoảng 3.000 km đường cao tốc...
URL: https://thoibaotaichinhvietnam.vn/stores/news_dataimages/dao...

 [综合类]
Caption: Các cấp Hội LHPN tỉnh Quảng Trị luôn đồng hành cùng hội viên trong các phong trà...
URL: https://daihoi13.dangcongsan.vn/Uploads/Images/2021/6/16/74/...

 [场景类]
Caption: MINI Countryman mới ra mắt tại Việt Nam ...
URL: https://cdn-i.vtcnews.vn/resize/th/upload/2021/03/12/image00...

 [综合类]
Caption: Khách ào ạt đặt hộp hoa nến làm quà 8/3, chủ hàng 'không kịp thở' ...
URL: https://cdn-i.vtcnews.vn/resize/th/upload/2023/03/01/image-m...

 [文化类]
Caption: Một số điệu múa cổ truyền của người Mơ Nông đã được khôi phục, truyền dạy cho lớ...
URL: https://media.moitruongvadothi.vn/images/2022/06/29/9869-165...


### xlm-roberta-base模型下载与加载

In [8]:
from mindnlp.transformers import AutoTokenizer, AutoModel

MODEL_ID = "FacebookAI/xlm-roberta-base"

logging.info(f"正在从 Hub 下载模型: {MODEL_ID} ...")
    
# 1. 下载并加载
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)
    
logging.info(f"模型下载完成！")

[WARNING] ME(9563:255086326722592,MainProcess):2026-04-22-01:15:27.480.278 [mindspore/context.py:1335] For 'context.set_context', the parameter 'ascend_config' will be deprecated and removed in a future version. Please use the api mindspore.device_context.ascend.op_precision.precision_mode(),
                                                       mindspore.device_context.ascend.op_precision.op_precision_mode(),
                                                       mindspore.device_context.ascend.op_precision.matmul_allow_hf32(),
                                                       mindspore.device_context.ascend.op_precision.conv_allow_hf32(),
                                                       mindspore.device_context.ascend.op_tuning.op_compile() instead.
Building prefix dict from the default dictionary ...
2026-04-22 01:15:34,222 [DEBUG] Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
2026-04-22 01:15:37,248 [DEBUG] Dumping mod

In [9]:
# 设置为推理模式
model.set_train(False)

XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear (768 -> 768)
            (key): Linear (768 -> 768)
            (value): Linear (768 -> 768)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear (768 -> 768)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): XLMRobertaIntermediate(
      

## 为数据集图像建立向量索引

In [11]:
import numpy as np
import time

def get_embedding(text, max_length=128):
    """提取文本句向量（[CLS] token），L2 归一化"""
    inputs = tokenizer(
        text, return_tensors='ms',
        max_length=max_length, truncation=True, padding='max_length',
    )
    outputs = model(**inputs)
    vec = outputs.last_hidden_state[0][0].asnumpy()
    return vec / (np.linalg.norm(vec) + 1e-8)


logging.info(f"开始建立索引，共 {len(all_data)} 条 caption...")
process_itemNum = 1000
t0 = time.time()
index = []

for i, item in enumerate(all_data):
    vec = get_embedding(item['caption'])
    index.append({'vec': vec, **item})
    if (i + 1) % 100 == 0:
        logging.info(f"  已处理 {i+1} / {len(all_data)} 条")
    if i >= process_itemNum: 
        break

logging.info(f"索引建立完成  共 {len(index)} 条  耗时 {time.time()-t0:.1f}s")

2026-04-22 01:19:30,650 [INFO] 开始建立索引，共 77486 条 caption...
2026-04-22 01:20:07,396 [INFO]   已处理 100 / 77486 条
2026-04-22 01:20:43,294 [INFO]   已处理 200 / 77486 条
2026-04-22 01:21:15,155 [INFO]   已处理 300 / 77486 条
2026-04-22 01:21:46,811 [INFO]   已处理 400 / 77486 条
2026-04-22 01:22:20,277 [INFO]   已处理 500 / 77486 条
2026-04-22 01:22:57,687 [INFO]   已处理 600 / 77486 条
2026-04-22 01:23:28,520 [INFO]   已处理 700 / 77486 条
2026-04-22 01:23:59,563 [INFO]   已处理 800 / 77486 条
2026-04-22 01:24:35,815 [INFO]   已处理 900 / 77486 条
2026-04-22 01:25:13,209 [INFO]   已处理 1000 / 77486 条
2026-04-22 01:25:13,547 [INFO] 索引建立完成  共 1001 条  耗时 342.9s


## Demo: 越南语图像内容搜索

In [12]:
def search(query, top_k=5):
    """输入越南语查询文本，返回 Top-K 最相关结果"""
    query_vec = get_embedding(query)
    scores = [
        (float(np.dot(query_vec, item['vec'])), item)
        for item in index
    ]
    scores.sort(key=lambda x: -x[0])
    return scores[:top_k]

In [13]:
# 输入越南语查询文本
query = "điện thoại thông minh màu xanh đặt trên bàn"

print(f"查询: {query}")
print('=' * 65)

# 返回语义最相关的图像及其越南语描述
results = search(query, top_k=6)
for rank, (score, item) in enumerate(results, 1):
    print(f"\nTop{rank}  相似度={score:.4f}  [{item['label']}]")
    print(f"描述: {item['caption'][:100]}...")
    print(f"图像: {item['url']}")

查询: điện thoại thông minh màu xanh đặt trên bàn

Top1  相似度=0.9997  [文化类]
描述: Hướng dẫn cách chụp màn hình Apple Watch nh...
图像: https://cdn-i.vtcnews.vn/resize/th/upload/2023/04/26/chup-man-hinh-apple-watch-1-14395627.jpeg

Top2  相似度=0.9996  [场景类]
描述: Lỗ hổng an ninh năng lượng ...
图像: https://cdn.vietnambiz.vn/2019/5/18/photo-2-155815466312795396684.jpg

Top3  相似度=0.9996  [场景类]
描述: Tài trợ kinh phí để báo chí chống gỗ lậu...
图像: https://thoibaotaichinhvietnam.vn/stores/news_dataimages/thoibaotaichinhvietnamvn/122015/16/16/medium/gl120210814092305.6459560.jpg?151216042100

Top4  相似度=0.9996  [文化类]
描述: gương mặt thân quen...
图像: https://cdn-i.vtcnews.vn/files/f2/2014/06/14/truc-tiep-chung-ket-guong-mat-than-quen-9.jpg

Top5  相似度=0.9996  [文化类]
描述: cha tự long...
图像: https://cdn-i.vtcnews.vn/files/f2/2015/07/11/tu-long-va-10-dieu-khan-gia-it-biet-1.jpg

Top6  相似度=0.9996  [综合类]
描述: Những loại lá bình thường bỗng thành hàng hot trên trang thương mại điện tử  ...
图像: https://cdn-i.vtcnews.vn/